## Exploración de los datos
Realiza un análisis exploratorio del conjunto de datos que incluya:
    • Dimensiones del dataset. 
    • Tipos de variables. 
    • Valores nulos. 
    • Registros duplicados. 
    • Estadísticos descriptivos. 
    • Visualizaciones que permitan comprender la información. 
    

In [11]:
import pandas as pd
import matplotlib.pyplot as plt

# Install dependencies as needed:
# pip install kagglehub[pandas-datasets]
import kagglehub
from kagglehub import KaggleDatasetAdapter

# Set the path to the file you'd like to load
file_path = "steam-200k.csv"


#%pip install "kagglehub[pandas-datasets]"


# Load the latest version
df_raw = kagglehub.dataset_load(
  KaggleDatasetAdapter.PANDAS,
  "tamber/steam-video-games",
  file_path,
  pandas_kwargs={"header": None}
  # Provide any additional arguments like 
  # sql_query or pandas_kwargs. See the 
  # documenation for more information:
  # https://github.com/Kaggle/kagglehub/blob/main/README.md#kaggledatasetadapterpandas
)

df = df_raw.copy()

print("First 5 records:", df.head())



First 5 records:            0                           1         2      3  4
0  151603712  The Elder Scrolls V Skyrim  purchase    1.0  0
1  151603712  The Elder Scrolls V Skyrim      play  273.0  0
2  151603712                   Fallout 4  purchase    1.0  0
3  151603712                   Fallout 4      play   87.0  0
4  151603712                       Spore  purchase    1.0  0


In [9]:
print(df.columns.tolist())

['151603712', 'The Elder Scrolls V Skyrim', 'purchase', '1.0', '0']


Podemos observar que la base de datos no contiene titulos de columna, por lo que pasaremos a proponer el titulo relevante

In [13]:
df.shape

(200000, 5)

El dataset contiene 200.000 registros y 5 columnas. Cada registro corresponde a una interacción de un usuario con un videojuego; por tanto, un mismo usuario puede aparecer en varias filas.

In [14]:
df[0].nunique()

12393

El conjunto de datos contiene 200.000 registros distribuidos en 5 columnas. La primera columna presenta 12.393 identificadores de usuario distintos. Por tanto, los registros corresponden a interacciones de esos usuarios con videojuegos, y no a 200.000 usuarios diferentes.

In [ ]:
['user_id', 'game', 'behavior_name', 'behavior_metric' ]

Realizamos una propuesta de titulo para categorizar cada columna segun lo que representan los datos, y se elimina la columna 5ta debido a que no aporta valor estadistico a los datos. 

In [15]:
df[4].value_counts(dropna=False)

4
0    200000
Name: count, dtype: int64

La quinta columna es constante: contiene el valor 0 en los 200.000 registros. Por tanto, no permite diferenciar las interacciones ni los usuarios y se excluirá de las variables utilizadas para la segmentación.

In [16]:
df = df.drop(columns=[4]) #Eliminamos columna sobrante

In [17]:
df.columns = ['user_id', 'game', 'behavior_name', 'behavior_metric'] #Asignamos los nombres a las columnas que conservamos

In [19]:
df.shape
df.head()

,user_id,game,behavior_name,behavior_metric
0,151603712,The Elder Scrolls V Skyrim,purchase,1.0
1,151603712,The Elder Scrolls V Skyrim,play,273.0
2,151603712,Fallout 4,purchase,1.0
3,151603712,Fallout 4,play,87.0
4,151603712,Spore,purchase,1.0


Pasamos a identificar los tipos de variables de los datos

In [20]:
df.dtypes

user_id              int64
game                object
behavior_name       object
behavior_metric    float64
dtype: object

In [21]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 200000 entries, 0 to 199999
Data columns (total 4 columns):
 #   Column           Non-Null Count   Dtype  
---  ------           --------------   -----  
 0   user_id          200000 non-null  int64  
 1   game             200000 non-null  object 
 2   behavior_name    200000 non-null  object 
 3   behavior_metric  200000 non-null  float64
dtypes: float64(1), int64(1), object(2)
memory usage: 6.1+ MB


No se identificaron valores nulos en las cuatro columnas analizadas, según la salida de df.info().

In [22]:
df.duplicated().sum() #Buscamos identificar dupicados

np.int64(707)

Se identificaron 707 filas que repiten los valores de las cuatro columnas de una fila anterior.

In [23]:
df[df.duplicated(keep=False)].sort_values(
    ['user_id', 'game', 'behavior_name', 'behavior_metric']
).head(10)

,user_id,game,behavior_name,behavior_metric
142904,561758,Sid Meier's Civilization IV,purchase,1.0
142905,561758,Sid Meier's Civilization IV,purchase,1.0
142906,561758,Sid Meier's Civilization IV Beyond the Sword,purchase,1.0
142907,561758,Sid Meier's Civilization IV Beyond the Sword,purchase,1.0
142908,561758,Sid Meier's Civilization IV Colonization,purchase,1.0
142909,561758,Sid Meier's Civilization IV Colonization,purchase,1.0
142910,561758,Sid Meier's Civilization IV Warlords,purchase,1.0
142911,561758,Sid Meier's Civilization IV Warlords,purchase,1.0
89867,1936551,Sid Meier's Civilization IV Colonization,purchase,1.0
90025,1936551,Sid Meier's Civilization IV Colonization,purchase,1.0


In [24]:
df.loc[df.duplicated(), 'behavior_name'].value_counts()


behavior_name
purchase    707
Name: count, dtype: int64

Se identificaron 707 filas duplicadas exactas, todas asociadas al comportamiento purchase. Estas repiten los valores de las cuatro columnas de un registro anterior. Su tratamiento se decidirá en la etapa de preparación de datos.

In [25]:
df[['game', 'behavior_name']].describe()

,game,behavior_name
count,200000,200000
unique,5155,2
top,Dota 2,purchase
freq,9682,129511


Se identificaron 5.155 videojuegos distintos. Dota 2 es el título con mayor presencia en el dataset, con 9.682 registros. El comportamiento más frecuente es purchase, con 129.511 registros. Estos conteos corresponden a filas, no a usuarios únicos.

In [26]:
df.groupby('behavior_name')['behavior_metric'].describe()

,count,mean,std,min,25%,50%,75%,max
behavior_name,,,,,,,,
play,70489.0,48.878063,229.335236,0.1,1.0,4.5,19.1,11754.0
purchase,129511.0,1.000000,0.000000,1.0,1.0,1.0,1.0,1.0


La media de horas por registro de juego (48,88) es considerablemente mayor que la mediana (4,5). Esta diferencia sugiere que los registros con muchas horas elevan el promedio, por lo que conviene complementar su interpretación con la mediana y los cuartiles.